In [1]:
# import pandas as pd
# import numpy as np
# import os
# from sklearn.cluster import DBSCAN

# # --- تنظیمات آدرس‌ها ---
# file_path = r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx'
# output_filename = r'outputs\G11\dsas_g11_lubrication_system_univariate\univariate\dsas_g11_univariate_output3.xlsx'

# target_sensors = ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']

# def remove_outliers_dbscan(series):
#     """حذف داده‌های پرت با استفاده از الگوریتم DBSCAN"""
#     if series.empty or len(series) < 5:  # حداقل داده مورد نیاز برای DBSCAN
#         return series
    
#     # تغییر شکل داده برای DBSCAN
#     X = series.values.reshape(-1, 1)
    
#     # محاسبه eps با مدیریت خطا
#     std_val = series.std()
    
#     # اگر انحراف معیار صفر یا NaN است، از مقدار پیش‌فرض استفاده کن
#     if pd.isna(std_val) or std_val == 0:
#         eps = 0.1  # مقدار پیش‌فرض
#     else:
#         eps = std_val * 0.5
#         # اطمینان از اینکه eps در محدوده مجاز است (بیشتر از 0)
#         eps = max(eps, 0.01)  # حداقل مقدار 0.01
    
#     # تنظیم پارامترها: eps فاصله همسایگی و min_samples حداقل تعداد همسایه
#     min_samples = min(5, len(series)-1)
#     min_samples = max(min_samples, 1)  # حداقل 1
    
#     dbscan = DBSCAN(eps=eps, min_samples=min_samples)
#     clusters = dbscan.fit_predict(X)
    
#     # فقط داده‌هایی که جزو کلاستر اصلی هستند (-1 نباشند)
#     # اگر همه داده‌ها outlier شدند، همه را برگردان
#     if (clusters == -1).all():
#         return series
    
#     return series[clusters != -1]

# def run_comprehensive_analysis():
#     if not os.path.exists(file_path):
#         print(f"❌ فایل یافت نشد.")
#         return

#     try:
#         df = pd.read_excel(file_path, parse_dates=['date'])
#         df = df.sort_values('date')
#     except Exception as e:
#         print(f"❌ خطا در خواندن فایل: {e}")
#         return

#     # ۱. تعریف بازه‌های زمانی
#     last_date = df['date'].max()
#     fault_start = last_date - pd.Timedelta(days=30)
#     baseline_start = fault_start - pd.Timedelta(days=30)

#     # جداسازی داده‌ها
#     df_baseline = df[(df['date'] >= baseline_start) & (df['date'] < fault_start)].copy()
#     df_fault = df[df['date'] >= fault_start].copy()

#     print(f"📊 بازه زمانی Baseline: {baseline_start.date()} تا {fault_start.date()}")
#     print(f"📊 بازه زمانی Fault: {fault_start.date()} تا {last_date.date()}")
#     print(f"📈 تعداد داده‌های Baseline: {len(df_baseline)}")
#     print(f"📈 تعداد داده‌های Fault: {len(df_fault)}")

#     # --- بخش اول: آماده‌سازی تب EWMA (داده‌های یک ماه اخیر) ---
#     ewma_list = []
#     baseline_stats = []

#     for col in target_sensors:
#         if col not in df.columns:
#             print(f"⚠️ سنسور {col} در فایل وجود ندارد")
#             continue

#         # الف) محاسبات تب اول (EWMA)
#         temp_fault = df_fault[['date', col]].copy()
#         if temp_fault.empty:
#             print(f"⚠️ داده‌ای برای سنسور {col} در بازه Fault وجود ندارد")
#         else:
#             temp_fault = temp_fault.rename(columns={col: 'Value'})
#             temp_fault['AssetID'] = col
#             temp_fault['EWMA'] = temp_fault['Value'].ewm(alpha=0.2, adjust=False).mean()
#             ewma_list.append(temp_fault[['date', 'AssetID', 'Value', 'EWMA']])

#         # ب) محاسبات تب دوم (Baseline با DBSCAN)
#         if col in df_baseline.columns:
#             baseline_series = df_baseline[col].dropna()
            
#             if len(baseline_series) < 5:
#                 print(f"⚠️ داده‌های Baseline سنسور {col} برای DBSCAN کافی نیست (تعداد: {len(baseline_series)})")
#                 clean_baseline = baseline_series
#             else:
#                 try:
#                     clean_baseline = remove_outliers_dbscan(baseline_series)
#                 except Exception as e:
#                     print(f"⚠️ خطا در حذف outliers برای سنسور {col}: {e}")
#                     clean_baseline = baseline_series  # در صورت خطا، داده اصلی را نگه دار

#             if not clean_baseline.empty:
#                 mean_val = clean_baseline.mean()
#                 std_val = clean_baseline.std()

#                 baseline_stats.append({
#                     'AssetID': col,
#                     'Clean_Mean': round(mean_val, 4),
#                     'Clean_Std': round(std_val, 4),
#                     'Normal_Upper_Limit(3Sigma)': round(mean_val + 3*std_val, 4),
#                     'Normal_Lower_Limit(3Sigma)': round(mean_val - 3*std_val, 4),
#                     'Outliers_Removed': len(df_baseline[col]) - len(clean_baseline),
#                     'Baseline_Data_Count': len(clean_baseline),
#                     'Original_Data_Count': len(df_baseline[col].dropna())
#                 })
#             else:
#                 print(f"⚠️ پس از پاکسازی، داده‌ای برای سنسور {col} باقی نمانده است")
                
#                 # اضافه کردن آمار با داده اصلی
#                 original_series = df_baseline[col].dropna()
#                 if not original_series.empty:
#                     mean_val = original_series.mean()
#                     std_val = original_series.std()
#                     baseline_stats.append({
#                         'AssetID': col,
#                         'Clean_Mean': round(mean_val, 4),
#                         'Clean_Std': round(std_val, 4),
#                         'Normal_Upper_Limit(3Sigma)': round(mean_val + 3*std_val, 4),
#                         'Normal_Lower_Limit(3Sigma)': round(mean_val - 3*std_val, 4),
#                         'Outliers_Removed': 0,
#                         'Baseline_Data_Count': len(original_series),
#                         'Original_Data_Count': len(original_series),
#                         'Note': 'Used original data (DBSCAN removed all points)'
#                     })

#     # --- ذخیره در اکسل با دو تب مجزا ---
#     try:
#         os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        
#         # بررسی وجود حداقل یک sheet برای ذخیره
#         has_ewma = len(ewma_list) > 0
#         has_stats = len(baseline_stats) > 0
        
#         if not has_ewma and not has_stats:
#             print("❌ هیچ داده‌ای برای ذخیره وجود ندارد!")
#             return
        
#         with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
#             # تب اول - EWMA
#             if has_ewma:
#                 df_ewma_final = pd.concat(ewma_list, ignore_index=True)
#                 df_ewma_final.to_excel(writer, sheet_name='EWMA_Comparison', index=False)
#                 print(f"✅ تب EWMA_Comparison با {len(df_ewma_final)} رکورد ایجاد شد")
#             else:
#                 # ایجاد یک sheet خالی با پیام توضیحی
#                 empty_df = pd.DataFrame({'Message': ['No EWMA data available for the fault period']})
#                 empty_df.to_excel(writer, sheet_name='EWMA_Comparison', index=False)
#                 print("⚠️ تب EWMA_Comparison خالی ایجاد شد")

#             # تب دوم - Baseline Stats
#             if has_stats:
#                 df_stats_final = pd.DataFrame(baseline_stats)
#                 df_stats_final.to_excel(writer, sheet_name='Baseline_Stats', index=False)
#                 print(f"✅ تب Baseline_Stats با {len(df_stats_final)} سنسور ایجاد شد")
#             else:
#                 # ایجاد یک sheet خالی با پیام توضیحی
#                 empty_df = pd.DataFrame({'Message': ['No baseline statistics available']})
#                 empty_df.to_excel(writer, sheet_name='Baseline_Stats', index=False)
#                 print("⚠️ تب Baseline_Stats خالی ایجاد شد")

#         print(f"\n🚀 گزارش با موفقیت ساخته شد:\n{output_filename}")
#         print("✅ Baseline_Stats با استفاده از DBSCAN پاکسازی و محاسبه شد.")
        
#     except Exception as e:
#         print(f"❌ خطا در ذخیره فایل: {e}")
#         import traceback
#         traceback.print_exc()

# # اجرای تابع
# run_comprehensive_analysis()

In [5]:
import pandas as pd
import numpy as np
import os
from sklearn.cluster import DBSCAN

# --- تنظیمات آدرس‌ها ---
file_path = r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_lubrication_system_univariate\univariate\dsas_g11_univariate_output3.xlsx'

target_sensors = ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']

def remove_outliers_dbscan(series):
    """حذف داده‌های پرت با استفاده از الگوریتم DBSCAN"""
    if series.empty or len(series) < 5:  # حداقل داده مورد نیاز برای DBSCAN
        return series
    
    # تغییر شکل داده برای DBSCAN
    X = series.values.reshape(-1, 1)
    
    # محاسبه eps با مدیریت خطا
    std_val = series.std()
    
    # اگر انحراف معیار صفر یا NaN است، از مقدار پیش‌فرض استفاده کن
    if pd.isna(std_val) or std_val == 0:
        eps = 0.1  # مقدار پیش‌فرض
    else:
        eps = std_val * 0.5
        # اطمینان از اینکه eps در محدوده مجاز است (بیشتر از 0)
        eps = max(eps, 0.01)  # حداقل مقدار 0.01
    
    # تنظیم پارامترها: eps فاصله همسایگی و min_samples حداقل تعداد همسایه
    min_samples = min(5, len(series)-1)
    min_samples = max(min_samples, 1)  # حداقل 1
    
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    clusters = dbscan.fit_predict(X)
    
    # فقط داده‌هایی که جزو کلاستر اصلی هستند (-1 نباشند)
    # اگر همه داده‌ها outlier شدند، همه را برگردان
    if (clusters == -1).all():
        return series
    
    return series[clusters != -1]

def run_comprehensive_analysis():
    if not os.path.exists(file_path):
        print(f"❌ فایل یافت نشد.")
        return

    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date')
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return

    # ۱. تعریف بازه‌های زمانی
    last_date = df['date'].max()
    fault_start = last_date - pd.Timedelta(days=30)
    baseline_start = fault_start - pd.Timedelta(days=30)

    # جداسازی داده‌ها
    df_baseline = df[(df['date'] >= baseline_start) & (df['date'] < fault_start)].copy()
    df_fault = df[df['date'] >= fault_start].copy()

    print(f"📊 بازه زمانی Baseline: {baseline_start.date()} تا {fault_start.date()}")
    print(f"📊 بازه زمانی Fault: {fault_start.date()} تا {last_date.date()}")
    print(f"📈 تعداد داده‌های Baseline: {len(df_baseline)}")
    print(f"📈 تعداد داده‌های Fault: {len(df_fault)}")

    # --- دیکشنری برای ذخیره میانگین و انحراف معیار هر سنسور (از داده‌های Baseline پاکسازی شده) ---
    sensor_stats = {}

    # --- ابتدا محاسبه آمار برای هر سنسور از داده‌های Baseline ---
    for col in target_sensors:
        if col not in df.columns:
            print(f"⚠️ سنسور {col} در فایل وجود ندارد")
            continue
            
        if col in df_baseline.columns:
            baseline_series = df_baseline[col].dropna()
            
            if len(baseline_series) < 5:
                print(f"⚠️ داده‌های Baseline سنسور {col} برای DBSCAN کافی نیست (تعداد: {len(baseline_series)})")
                clean_baseline = baseline_series
            else:
                try:
                    clean_baseline = remove_outliers_dbscan(baseline_series)
                except Exception as e:
                    print(f"⚠️ خطا در حذف outliers برای سنسور {col}: {e}")
                    clean_baseline = baseline_series
            
            if not clean_baseline.empty:
                mean_val = clean_baseline.mean()
                std_val = clean_baseline.std()
                sensor_stats[col] = {
                    'mean': round(mean_val, 4), 
                    'std': round(std_val, 4),
                    'upper_band': round(mean_val + 3 * std_val, 4),
                    'lower_band': round(mean_val - 3 * std_val, 4)
                }
                print(f"✅ سنسور {col}: میانگین={round(mean_val, 4)}, انحراف معیار={round(std_val, 4)}, "
                      f"upper_band={round(mean_val + 3*std_val, 4)}, lower_band={round(mean_val - 3*std_val, 4)}")
            else:
                # استفاده از داده اصلی در صورت خالی بودن داده پاکسازی شده
                original_series = df_baseline[col].dropna()
                if not original_series.empty:
                    mean_val = original_series.mean()
                    std_val = original_series.std()
                    sensor_stats[col] = {
                        'mean': round(mean_val, 4), 
                        'std': round(std_val, 4),
                        'upper_band': round(mean_val + 3 * std_val, 4),
                        'lower_band': round(mean_val - 3 * std_val, 4)
                    }
                    print(f"⚠️ سنسور {col}: استفاده از داده اصلی (میانگین={round(mean_val, 4)})")
        else:
            print(f"⚠️ سنسور {col} در داده‌های Baseline وجود ندارد")

    # --- بخش دوم: آماده‌سازی تب EWMA (داده‌های یک ماه اخیر) با افزودن ستون‌های میانگین، انحراف معیار و باندها ---
    ewma_list = []
    baseline_stats = []

    for col in target_sensors:
        if col not in df.columns:
            continue

        # الف) محاسبات تب اول (EWMA) با افزودن ستون‌های تکراری
        temp_fault = df_fault[['date', col]].copy()
        if temp_fault.empty:
            print(f"⚠️ داده‌ای برای سنسور {col} در بازه Fault وجود ندارد")
        else:
            temp_fault = temp_fault.rename(columns={col: 'Value'})
            temp_fault['AssetID'] = col
            temp_fault['EWMA'] = temp_fault['Value'].ewm(alpha=0.2, adjust=False).mean()
            
            # افزودن ستون‌های میانگین، انحراف معیار و باندهای بالا و پایین
            if col in sensor_stats:
                temp_fault['Repeated_Mean'] = sensor_stats[col]['mean']
                temp_fault['Repeated_Std'] = sensor_stats[col]['std']
                temp_fault['upper_band'] = sensor_stats[col]['upper_band']
                temp_fault['lower_band'] = sensor_stats[col]['lower_band']
            else:
                temp_fault['Repeated_Mean'] = np.nan
                temp_fault['Repeated_Std'] = np.nan
                temp_fault['upper_band'] = np.nan
                temp_fault['lower_band'] = np.nan
            
            ewma_list.append(temp_fault[['date', 'AssetID', 'Value', 'EWMA', 'Repeated_Mean', 'Repeated_Std', 'upper_band', 'lower_band']])

        # ب) محاسبات تب دوم (Baseline با DBSCAN) - بدون تغییر
        if col in df_baseline.columns:
            baseline_series = df_baseline[col].dropna()
            
            if len(baseline_series) < 5:
                clean_baseline = baseline_series
            else:
                try:
                    clean_baseline = remove_outliers_dbscan(baseline_series)
                except Exception as e:
                    clean_baseline = baseline_series

            if not clean_baseline.empty:
                mean_val = clean_baseline.mean()
                std_val = clean_baseline.std()

                baseline_stats.append({
                    'AssetID': col,
                    'Clean_Mean': round(mean_val, 4),
                    'Clean_Std': round(std_val, 4),
                    'Normal_Upper_Limit(3Sigma)': round(mean_val + 3*std_val, 4),
                    'Normal_Lower_Limit(3Sigma)': round(mean_val - 3*std_val, 4),
                    'Outliers_Removed': len(df_baseline[col]) - len(clean_baseline),
                    'Baseline_Data_Count': len(clean_baseline),
                    'Original_Data_Count': len(df_baseline[col].dropna())
                })
            else:
                original_series = df_baseline[col].dropna()
                if not original_series.empty:
                    mean_val = original_series.mean()
                    std_val = original_series.std()
                    baseline_stats.append({
                        'AssetID': col,
                        'Clean_Mean': round(mean_val, 4),
                        'Clean_Std': round(std_val, 4),
                        'Normal_Upper_Limit(3Sigma)': round(mean_val + 3*std_val, 4),
                        'Normal_Lower_Limit(3Sigma)': round(mean_val - 3*std_val, 4),
                        'Outliers_Removed': 0,
                        'Baseline_Data_Count': len(original_series),
                        'Original_Data_Count': len(original_series),
                        'Note': 'Used original data (DBSCAN removed all points)'
                    })

    # --- ذخیره در اکسل با دو تب مجزا ---
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        
        has_ewma = len(ewma_list) > 0
        has_stats = len(baseline_stats) > 0
        
        if not has_ewma and not has_stats:
            print("❌ هیچ داده‌ای برای ذخیره وجود ندارد!")
            return
        
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            # تب اول - EWMA (با ستون‌های جدید)
            if has_ewma:
                df_ewma_final = pd.concat(ewma_list, ignore_index=True)
                df_ewma_final.to_excel(writer, sheet_name='EWMA_Comparison', index=False)
                print(f"✅ تب EWMA_Comparison با {len(df_ewma_final)} رکورد ایجاد شد")
                print(f"   📋 ستون‌های موجود: date, AssetID, Value, EWMA, Repeated_Mean, Repeated_Std, upper_band, lower_band")
                print(f"   📊 upper_band = Repeated_Mean + 3 * Repeated_Std")
                print(f"   📊 lower_band = Repeated_Mean - 3 * Repeated_Std")
            else:
                empty_df = pd.DataFrame({'Message': ['No EWMA data available for the fault period']})
                empty_df.to_excel(writer, sheet_name='EWMA_Comparison', index=False)
                print("⚠️ تب EWMA_Comparison خالی ایجاد شد")

            # تب دوم - Baseline Stats (بدون تغییر)
            if has_stats:
                df_stats_final = pd.DataFrame(baseline_stats)
                df_stats_final.to_excel(writer, sheet_name='Baseline_Stats', index=False)
                print(f"✅ تب Baseline_Stats با {len(df_stats_final)} سنسور ایجاد شد")
            else:
                empty_df = pd.DataFrame({'Message': ['No baseline statistics available']})
                empty_df.to_excel(writer, sheet_name='Baseline_Stats', index=False)
                print("⚠️ تب Baseline_Stats خالی ایجاد شد")

        print(f"\n🚀 گزارش با موفقیت ساخته شد:\n{output_filename}")
        print("✅ در تب اول، ستون‌های زیر به ازای هر ردیف تکرار شده‌اند:")
        print("   - Repeated_Mean: میانگین داده‌های پاکسازی شده")
        print("   - Repeated_Std: انحراف معیار داده‌های پاکسازی شده")
        print("   - upper_band: حد بالایی (میانگین + 3 × انحراف معیار)")
        print("   - lower_band: حد پایینی (میانگین - 3 × انحراف معیار)")
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")
        import traceback
        traceback.print_exc()

# اجرای تابع
run_comprehensive_analysis()

📊 بازه زمانی Baseline: 2026-03-27 تا 2026-04-26
📊 بازه زمانی Fault: 2026-04-26 تا 2026-05-26
📈 تعداد داده‌های Baseline: 55
📈 تعداد داده‌های Fault: 103
✅ سنسور AssetID_9375: میانگین=51.0, انحراف معیار=0.0, upper_band=51.0, lower_band=51.0
✅ سنسور AssetID_8341: میانگین=0.1137, انحراف معیار=0.0201, upper_band=0.174, lower_band=0.0534
✅ سنسور AssetID_8343: میانگین=68.8909, انحراف معیار=0.5331, upper_band=70.4902, lower_band=67.2917
✅ سنسور AssetID_8344: میانگین=-240.9091, انحراف معیار=2.9013, upper_band=-232.2052, lower_band=-249.613
✅ سنسور AssetID_8346: میانگین=6.0, انحراف معیار=0.0, upper_band=6.0, lower_band=6.0
✅ سنسور AssetID_9286: میانگین=7.5906, انحراف معیار=0.0295, upper_band=7.6791, lower_band=7.502
✅ سنسور AssetID_9287: میانگین=1.22, انحراف معیار=0.0, upper_band=1.22, lower_band=1.22
✅ تب EWMA_Comparison با 721 رکورد ایجاد شد
   📋 ستون‌های موجود: date, AssetID, Value, EWMA, Repeated_Mean, Repeated_Std, upper_band, lower_band
   📊 upper_band = Repeated_Mean + 3 * Repeated_Std
   